In [ ]:
# Module 2 - Lab 1: Train and Evaluate a Keras-Based Classifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'train_data',           # Change to your training folder
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'train_data',
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Model Definition
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Training
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator
)

# Evaluation & Plots
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

# Save model
model.save('keras_classifier.h5')
print("Keras model training completed!")

In [ ]:
# Module 2 - Lab 2: Implement and Test a PyTorch-Based Classifier

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Transforms
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder('train_data', transform=transform)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4)

# CNN Model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 512),  # Adjust size based on input
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=len(train_dataset.classes))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
def train_model(epochs=15):
    train_acc, val_acc = [], []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc.append(100. * correct / total)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc[-1]:.2f}%')
    
    torch.save(model.state_dict(), 'pytorch_classifier.pth')
    print("PyTorch model saved!")

train_model()

In [ ]:
# Module 2 - Lab 3: Comparative Analysis

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Example results (replace with your actual results)
comparison = {
    'Framework': ['Keras/TensorFlow', 'PyTorch'],
    'Final_Accuracy': [92.5, 91.8],
    'Final_Val_Accuracy': [88.7, 89.2],
    'Training_Time_min': [45, 38],
    'Parameters_M': [8.9, 8.7],
    'Ease_of_Use': [9.0, 8.5]   # Subjective score
}

df = pd.DataFrame(comparison)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(x='Framework', y='Final_Val_Accuracy', data=df, ax=axes[0,0])
axes[0,0].set_title('Validation Accuracy Comparison')

sns.barplot(x='Framework', y='Training_Time_min', data=df, ax=axes[0,1])
axes[0,1].set_title('Training Time (minutes)')

sns.barplot(x='Framework', y='Parameters_M', data=df, ax=axes[1,0])
axes[1,0].set_title('Number of Parameters (Millions)')

# Add your own observations
print("=== Comparative Analysis ===")
print(df)

print("\nObservations:")
print("• PyTorch was slightly faster to train")
print("• Keras had simpler high-level API")
print("• Both achieved very similar accuracy")
print("• PyTorch offers more flexibility for custom layers")